In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Wed Sep  2 11:58:35 PDT 2026


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils, gbd_data

In [3]:
from lsff_utils import gbd_data

In [4]:
location = "india"
vehicle = "rice"

In [5]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [6]:
intervention_scenarios = config_utils.get_config()["custom_intervention_scenarios"].get(
    location, ["intervention"]
)
intervention_scenarios

['intervention']

In [7]:
# Get most recent GBD year, used by calls in gbd_data
YEAR = gbd_data.most_recent_year()
YEAR

2023

## Forecasted births and stillbirths

In [8]:
# NOTE: Year must match base year used in TFR rescaling below, so we
# keep it at 2022 for now
# TODO: Update year from 2022 (to 2023 to match GBD?) if we can replace
# TFR data 
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate,
        "estimate",
        location.title(),
        years=2022,
    ).value

In [9]:
# Filter out lower and upper values, keep mean only
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr.loc[asfr>0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2022        2023        0.005896
                  15.0       20.0     2022        2023        0.073573
                  20.0       25.0     2022        2023        0.199620
                  25.0       30.0     2022        2023        0.214820
                                                                ...   
                  35.0       40.0     2022        2023        0.131014
                  40.0       45.0     2022        2023        0.069464
                  45.0       50.0     2022        2023        0.031685
                  50.0       55.0     2022        2023        0.002873
Name: value, Length: 9, dtype: float64

In [10]:
# TODO: Update this with more recent GBD data. Also, does this belong in
# 0100_data_prep? Maybe not, if it's only used in this notebook.
# NOTE: The year in the denominator here must match the year of the ASFR
# call above. The year in the numerator is the target year we want data
# for.
# Scale ASFR in each category down proportionally to the scale-down in
# total fertility rate (TFR) forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2022        2023        0.005266
                  15.0       20.0     2022        2023        0.065712
                  20.0       25.0     2022        2023        0.178290
                  25.0       30.0     2022        2023        0.191866
                                                                ...   
                  35.0       40.0     2022        2023        0.117014
                  40.0       45.0     2022        2023        0.062042
                  45.0       50.0     2022        2023        0.028299
                  50.0       55.0     2022        2023        0.002566
Name: value, Length: 9, dtype: float64

In [11]:
# Values now represent 2030 instead of 2022
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
Nigeria   Female  0.000000   0.019178   2030        2031        0.000000
                  0.019178   0.076712   2030        2031        0.000000
                  0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                                                                  ...   
                  35.000000  40.000000  2030        2031        0.117014
                  30.000000  35.000000  2030        2031        0.176792
                  20.000000  25.000000  2030        2031        0.178290
                  25.000000  30.000000  2030        2031        0.191866
Name: value, Length: 50, dtype: float64

In [12]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import resolve_location
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [13]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = resolve_location(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        # NOTE: Using RELEASE_IDS.GBD_2023 returns an empty dataframe,
        # so we're keeping GBD 2021 forecasts for now
        # TODO: Update to the latest GBD release once it provides 
        # forecasted population data
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [14]:
with gbd_data.quiet_gbd_logs():
    pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2030        2031         76752.699967
                  0.019178   0.076712    2030        2031        227071.059445
                  0.076712   0.500000    2030        2031                  NaN
                  0.500000   1.000000    2030        2031                  NaN
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        354428.130049
                  85.000000  90.000000   2030        2031        177910.014136
                  90.000000  95.000000   2030        2031         62189.478603
                  95.000000  125.000000  2030        2031         16933.452258
Name: value, Length: 50, dtype: float64

In [15]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [16]:
pop = pop.fillna(0)

In [17]:
n_births = (pop * asfr).sum()
n_births

9745557.876500718

In [18]:
# NOTE: The stillbirth ratio (SBR) pulled from GBD here will be applied
# to the future population in 2030 (or whatever target year is being
# used). The SBR does not vary much by year, so we just use data for the
# most recent GBD year (defined in YEAR above) and extrapolate that
# value forward to the target year.
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio,
        "estimate",
        location.title(),
        years=YEAR,
    ).value
sbr

location  year_start  year_end  parameter  
Nigeria   2023        2024      lower_value    0.020725
                                mean_value     0.026341
                                upper_value    0.032816
Name: value, dtype: float64

In [19]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
).squeeze()
sbr

0.0263411623194878

In [20]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

10.002267198419586

## Fertility (technically birth-and-stillbirth) disparities

In [21]:
# TODO: Update this DHS data, and move it into 0100_data_prep? We already
# have some stuff in there to process DHS data.
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        {  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            1: 56_979,
            2: 50_335,
            3: 45_189,
            4: 42_611,
            5: 36_290,
        }
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        {  # Table 8.4 Perinatal mortality
            1: 7_712,
            2: 7_886,
            3: 7_139,
            4: 6_328,
            5: 5_558,
        }
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        {  # Table 8.4 Perinatal mortality
            1: 2_645,
            2: 2_516,
            3: 2_290,
            4: 2_018,
            5: 1_592,
        }
    )

dist_births_and_stillbirths_by_wealth.index.name = "wealth_quintile"

# TODO: Clarify variable names: What does the prefix s_ stand for in
# this notebook? Claude thinks it means "stratified," but that it is
# applied inconsistently and sometimes redundantly with the suffix
# _by_wealth.
s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

wealth_quintile
1    2.170746e+06
2    2.219723e+06
3    2.009460e+06
4    1.781183e+06
5    1.564446e+06
dtype: float64

In [22]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

wealth_quintile
1    2.227926e+06
2    2.278193e+06
3    2.062392e+06
4    1.828101e+06
5    1.605655e+06
dtype: float64

In [23]:
# TODO: Update this with newer GBD data
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths in 2030 for under-1 year olds
# from GBD 2021
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate_per_birth = ntd_deaths / n_births
10_000 * ntd_death_rate_per_birth

5.513814671356135

In [24]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

21.13628957353185

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [25]:
# TODO: Search for updated folate intake data
if location == "india":
    folate_intake_by_wealth = pd.Series(
        {
            1: 220,
            2: 220,
            3: 220,
            4: 220,
            5: 220,  # NRV is 400 mcg/day
        }
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    folate_intake_by_wealth = pd.Series(
        {
            1: 189,
            2: 198,
            3: 197,
            4: 203,
            5: 208,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    folate_intake_by_wealth = pd.Series(
        {
            1: 166,
            2: 152,
            3: 137,
            4: 350,
            5: 469,  # NRV is 400 mcg/day
        }
    )  # how can we use this to estimate disparities in NTD?

folate_intake_by_wealth.index.name = "wealth_quintile"

In [26]:
# TODO: See if we can find better data here
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        {  # it includes wealth stratification, but has a very low threshold for insufficiency
            1: 1,  # so I am assuming that most everyone is in the danger zone for low folate
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        {  # can CHAMPS offer more detail?  Need to infer wealth somehow
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        {  # supplementation studies don't make this easy, but here is a guess
            1: 1,
            2: 1,
            3: 1,
            4: 1,
            5: 1,
        }
    )
    # FIXME: Why is this normalization only applied for Ethiopia? Should
    # it be applied for all locations? Also, Claude thinks the correct
    # normalization would be to divide by the bifth-weighted mean, not
    # the unweighted mean, and that the assert statement below only
    # passes because these values are all 1s.
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth.index.name = "wealth_quintile"
s_dist_deaths_by_wealth

wealth_quintile
1    1
2    1
3    1
4    1
5    1
dtype: int64

In [27]:
s_ntd_death_rate_per_birth = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate_per_birth

wealth_quintile
1    5.513815
2    5.513815
3    5.513815
4    5.513815
5    5.513815
dtype: float64

In [28]:
s_ntd_death_count = s_ntd_death_rate_per_birth * s_births
s_ntd_death_count

wealth_quintile
1    1196.909171
2    1223.914124
3    1107.979068
4     982.111156
5     862.606480
dtype: float64

In [29]:
assert np.isclose(s_ntd_death_count.sum(), ntd_deaths)

In [30]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

wealth_quintile
1    3391.242652
2    3467.756685
3    3139.274027
4    2782.648276
5    2444.051694
dtype: float64

In [31]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

wealth_quintile
1    4588.151824
2    4691.670809
3    4247.253095
4    3764.759432
5    3306.658174
dtype: float64

In [32]:
# NOTE: NTD risk here means risk of having an "NTD-affected pregnancy",
# which is a stillbirth due to NTD, OR a birth with NTD (not necessarily fatal!)
# See Kirke 1993 ("Maternal plasma folate and vitamin B12 are independent risk factors for neural tube defects")
# where it says: "Early foetal
# deaths (<23 weeks gestation) attributable to NTDs
# were excluded because of the incomplete ascertain-
# ment of such cases and the difficulty of obtaining a
# valid control group."
# This implies that late foetal deaths, roughly equivalent to stillbirths,
# are included.
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc

In [33]:
# TODO: Update these to GBD 2023
# From GBD 2021 using GBD Compare, for year 2021
# NTD incident cases / NTD deaths in <1 year olds
# (all of GBD's incident cases are those who survived birth)
if location == "india":
    ntd_death_to_live_birth_case_ratio = 11_796.34 / 5_492.68
elif location == "nigeria":
    ntd_death_to_live_birth_case_ratio = 21_756.56 / 6_231.21
elif location == "ethiopia":
    ntd_death_to_live_birth_case_ratio = 3_982.63 / 2_255.6

ntd_death_to_live_birth_case_ratio

3.4915465856551138

In [34]:
s_ntd_live_birth_cases = s_ntd_death_count * ntd_death_to_live_birth_case_ratio
s_ntd_live_birth_cases

wealth_quintile
1    4179.064131
2    4273.353181
3    3868.560533
4    3429.086854
5    3011.830710
dtype: float64

In [35]:
s_ntd_affected_pregnancies = s_ntd_stillbirth_count + s_ntd_live_birth_cases

In [36]:
ntd_affected_pregnancy_risk = (
    s_ntd_affected_pregnancies / s_births_and_stillbirths_by_wealth
)
ntd_affected_pregnancy_risk

wealth_quintile
1    0.003398
2    0.003398
3    0.003398
4    0.003398
5    0.003398
dtype: float64

In [37]:
s_ntd_death_or_stillbirth_count / s_births

wealth_quintile
1    0.002114
2    0.002114
3    0.002114
4    0.002114
5    0.002114
dtype: float64

In [38]:
backcalc_rbc(ntd_affected_pregnancy_risk, "daly")

wealth_quintile
1    410.669264
2    410.669264
3    410.669264
4    410.669264
5    410.669264
dtype: float64

In [39]:
backcalc_rbc(
    ntd_affected_pregnancy_risk, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9
# NOTE: It's a bit hard to compare, due to units issues. That table reports
# proportions under 151 ng/ml. In our units (nmol/L), that is ~342.
# https://www.wolframalpha.com/input?i=151+ng%2Fml+of+folate+to+nmol%2FL

wealth_quintile
1    415.761281
2    415.761281
3    415.761281
4    415.761281
5    415.761281
dtype: float64

In [40]:
pop

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2030        2031         76752.699967
                  0.019178   0.076712    2030        2031        227071.059445
                  0.076712   0.500000    2030        2031             0.000000
                  0.500000   1.000000    2030        2031             0.000000
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        354428.130049
                  85.000000  90.000000   2030        2031        177910.014136
                  90.000000  95.000000   2030        2031         62189.478603
                  95.000000  125.000000  2030        2031         16933.452258
Name: value, Length: 50, dtype: float64

In [41]:
s_pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
s_pop = s_pop.set_index([c for c in s_pop.columns if c != "value"])
s_pop

value
sex    age_start age_end    pregnant     wealth_quintile              
Female 0.0       0.019178   not_pregnant 1                17452.628925
                                         2                17380.997496
                                         3                16409.683914
                                         4                14424.706826
...                                                                ...
Male   95.0      125.000000 not_pregnant 2                 4424.735951
                                         3                 4548.244848
                                         4                 4870.219344
                                         5                 5193.002126

[285 rows x 1 columns]

In [42]:
# WRA only
s_pop = s_pop[
    (s_pop.index.get_level_values("sex") == "Female")
    & (s_pop.index.get_level_values("age_start") >= 15)
    & (s_pop.index.get_level_values("age_end") <= 50)
].copy()

In [43]:
s_daily_vehicle = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/amount/mean/{location}.csv"
)
assert (s_daily_vehicle.vehicle_name == vehicle).all()
s_daily_vehicle = s_daily_vehicle.drop(columns=["vehicle_name"])
s_daily_vehicle = s_daily_vehicle.set_index(
    [c for c in s_daily_vehicle.columns if c != "value"]
).value
s_daily_vehicle

sex     age_start  age_end  wealth_quintile
Female  0          5        1                  10.665035
                            2                  21.787441
                            3                  30.759427
                            4                  43.298027
                                                 ...    
Male    15         125      2                  27.112203
                            3                  38.276907
                            4                  53.879890
                            5                  62.068225
Name: value, Length: 30, dtype: float64

In [44]:
from lsff_utils import data_processing

In [45]:
s_daily_vehicle = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, s_daily_vehicle)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
s_daily_vehicle

wealth_quintile
1    11.4912
2    23.4752
3    33.1422
4    46.6521
5    53.7420
Name: value, dtype: float64

In [46]:
any_consumers = pd.read_csv(
    f"../0100_data_prep/results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
assert (any_consumers.vehicle_name == vehicle).all()
any_consumers = any_consumers.drop(columns=["vehicle_name"])
any_consumers = any_consumers.set_index(
    [c for c in any_consumers.columns if c != "value"]
).value
any_consumers

wealth_quintile  sex     age_start  age_end
1                Female  15         50         0.336
2                Female  15         50         0.448
3                Female  15         50         0.546
4                Female  15         50         0.633
5                Female  15         50         0.676
Name: value, dtype: float64

In [47]:
any_consumers = (
    s_pop.value
    * data_processing.reindex_series_onto_df_by_age_groups(s_pop, any_consumers)
).groupby("wealth_quintile").sum() / s_pop.value.groupby("wealth_quintile").sum()
any_consumers

wealth_quintile
1    0.336
2    0.448
3    0.546
4    0.633
5    0.676
Name: value, dtype: float64

In [48]:
s_daily_vehicle_among_consumers = s_daily_vehicle / any_consumers
s_daily_vehicle_among_consumers

wealth_quintile
1    34.2
2    52.4
3    60.7
4    73.7
5    79.5
Name: value, dtype: float64

In [49]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.0

In [50]:
intervention_concentration_mcg_per_gram = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.drop(columns=["vehicle_name"])
    .set_index("scenario")
    .value
)
intervention_concentration_mcg_per_gram

scenario
intervention    1.69
Name: value, dtype: float64

In [51]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()

In [52]:
if location == "india" and vehicle == "rice":
    # Confusingly, our baseline scenario (our best guess about the present)
    # is *not* a good guess about 2019-2020 (which is when our baseline folate estimate is from),
    # because this program has rolled out entirely since then:
    # In the phase-I of the roll out, the fortified rice was introduced in the social welfare schemes such as Integrated Child Development Scheme (ICDS)
    # and Pradhan Mantri Poshan Shakti Nirman (PM POSHAN, earlier known as the National Program of Mid-Day Meal in Schools)
    # throughout India during 2021–22 [18].
    # Phase-II has covered aspirational and high burden districts for anemia (total 291 districts) under Public Distribution System (PDS) and other welfare schemes,
    # in addition to Phase-I districts, by March 2023 [18].
    # All the remaining districts in India will be covered in Phase III by March 2024 [19].
    # ~ https://pmc.ncbi.nlm.nih.gov/articles/PMC11305529/
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline.assign(value=0)
else:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline

In [53]:
df_eff_fort_intervention = pd.concat(
    [
        pd.read_csv(
            f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"
        ).assign(scenario=intervention_scenario)
        for intervention_scenario in intervention_scenarios
    ]
)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()
df_eff_fort_intervention = df_eff_fort_intervention.drop(columns=["vehicle_name"])
df_eff_fort_intervention

,wealth_quintile,sex,value,scenario
0,1,Female,0.081648,intervention
1,1,Male,0.081648,intervention
2,2,Female,0.108864,intervention
3,2,Male,0.108864,intervention
...,...,...,...,...
6,4,Female,0.153819,intervention
7,4,Male,0.153819,intervention
8,5,Female,0.164268,intervention
9,5,Male,0.164268,intervention


In [54]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,1,2.538048e+06
41,Female,15.0,20.0,2,2.988632e+06
42,Female,15.0,20.0,3,2.997132e+06
43,Female,15.0,20.0,4,3.244674e+06
...,...,...,...,...,...
71,Female,45.0,50.0,2,8.013029e+05
72,Female,45.0,50.0,3,8.513975e+05
73,Female,45.0,50.0,4,9.617532e+05
74,Female,45.0,50.0,5,1.138775e+06


In [55]:
if "sex" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.sex == "Female")
    ]

if "age_start" in df_eff_fort_baseline_2019_2020.columns:
    df_eff_fort_baseline_2019_2020 = df_eff_fort_baseline_2019_2020[
        (df_eff_fort_baseline_2019_2020.age_start >= 15)
        & (df_eff_fort_baseline_2019_2020.age_end <= 50)
    ]

In [56]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [57]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population) * (
        1
        if "scenario" not in effective_fort.columns
        else effective_fort.scenario.nunique()
    )
    group_cols = [c for c in ["scenario", "wealth_quintile"] if c in merged]
    return merged.groupby(group_cols).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [58]:
df_eff_fort_baseline_2019_2020 = aggregate_using_population(
    df_eff_fort_baseline_2019_2020
)
df_eff_fort_baseline_2019_2020

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          rice                1         0.0     40  Female       15.0   
1          rice                1         0.0     45  Female       20.0   
2          rice                1         0.0     50  Female       25.0   
3          rice                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         rice                5         0.0     59  Female       30.0   
32         rice                5         0.0     64  Female       35.0   
33         rice                5         0.0     69  Female       40.0   
34         rice                5         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  2.538048e+06  
1      25.0  2.206803e+06  
2      30.0  1.805742e+06  
3      35.0  1.479843e+06  
..      ...           ...  
31     35.0  1.945402e+06  
32     40.0  1.615082e+06  
33     45.0  1.337273e+06  
34     

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [59]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

   vehicle_name  wealth_quintile  value_fort  index     sex  age_start  \
0          rice                1         0.0     40  Female       15.0   
1          rice                1         0.0     45  Female       20.0   
2          rice                1         0.0     50  Female       25.0   
3          rice                1         0.0     55  Female       30.0   
..          ...              ...         ...    ...     ...        ...   
31         rice                5         0.0     59  Female       30.0   
32         rice                5         0.0     64  Female       35.0   
33         rice                5         0.0     69  Female       40.0   
34         rice                5         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  2.538048e+06  
1      25.0  2.206803e+06  
2      30.0  1.805742e+06  
3      35.0  1.479843e+06  
..      ...           ...  
31     35.0  1.945402e+06  
32     40.0  1.615082e+06  
33     45.0  1.337273e+06  
34     

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [60]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

    wealth_quintile     sex  value_fort      scenario  index  age_start  \
0                 1  Female    0.081648  intervention     40       15.0   
1                 1  Female    0.081648  intervention     45       20.0   
2                 1  Female    0.081648  intervention     50       25.0   
3                 1  Female    0.081648  intervention     55       30.0   
..              ...     ...         ...           ...    ...        ...   
31                5  Female    0.164268  intervention     59       30.0   
32                5  Female    0.164268  intervention     64       35.0   
33                5  Female    0.164268  intervention     69       40.0   
34                5  Female    0.164268  intervention     74       45.0   

    age_end     value_pop  
0      20.0  2.538048e+06  
1      25.0  2.206803e+06  
2      30.0  1.805742e+06  
3      35.0  1.479843e+06  
..      ...           ...  
31     35.0  1.945402e+06  
32     40.0  1.615082e+06  
33     45.0  1.337273e+06

scenario      wealth_quintile
intervention  1                  0.081648
              2                  0.108864
              3                  0.132678
              4                  0.153819
              5                  0.164268
dtype: float64

In [61]:
RBC_baseline = backcalc_rbc(ntd_affected_pregnancy_risk, "crider")
RBC_baseline

wealth_quintile
1    415.761281
2    415.761281
3    415.761281
4    415.761281
5    415.761281
dtype: float64

In [62]:
# TODO: Document this value better
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [63]:
# TODO: Determine whether we still want the same fortification
# scenarios, and update baseline values
# Delete fortification effect baked into our baseline folate estimate.
# In non-India locations, this is going to be zero.
# For India, our current source for baseline folate is very rough,
# but it does appear to be from before the fortification program (2019-2020).
s_zero_folate = folate_intake_by_wealth - (
    df_eff_fort_baseline_2019_2020
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)

In [64]:
s_baseline_folate = s_zero_folate + (
    df_eff_fort_baseline
    * s_daily_vehicle_among_consumers
    * baseline_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_baseline_folate

wealth_quintile
1    189.0
2    198.0
3    197.0
4    203.0
5    208.0
dtype: float64

In [65]:
s_intervention_folate = s_zero_folate + (
    df_eff_fort_intervention
    * s_daily_vehicle_among_consumers
    * intervention_concentration_mcg_per_gram
    * fortification_mcg_to_dfe
)
s_intervention_folate

scenario      wealth_quintile
intervention  1                  197.022455
              2                  214.388953
              3                  220.137862
              4                  235.569650
              5                  245.519386
dtype: float64

In [66]:
zero_folate_pct_decrease = (
    folate_intake_by_wealth - s_zero_folate
) / folate_intake_by_wealth

In [67]:
baseline_folate_pct_increase_from_zero = (
    s_baseline_folate - s_zero_folate
) / folate_intake_by_wealth
baseline_folate_pct_increase_from_zero

wealth_quintile
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
dtype: float64

In [68]:
intevention_folate_pct_increase_from_zero = (
    s_intervention_folate - s_zero_folate
) / folate_intake_by_wealth
intevention_folate_pct_increase_from_zero

scenario      wealth_quintile
intervention  1                  0.042447
              2                  0.082772
              3                  0.117451
              4                  0.160442
              5                  0.180382
dtype: float64

In [69]:
# TODO: What is this hardcoded (1 + ((6 / 10) * pct_decrease) in this
# cell and the following? Claude says this line (and the subsequent
# cells) is converting a change in dietary folate intake into a change
# in RBC folate concentration using a proportional (elasticity) model
# with an elasticity of 0.6 (e.g. a 10% rise in dietary folate intake
# would lead to a 6% rise in RBC folate concentration). But the value of
# 0.6 is not documented anywhere. Does it come from Crider et al.?
RBC_zero = RBC_baseline / (1 + ((6 / 10) * zero_folate_pct_decrease))
RBC_zero

wealth_quintile
1    415.761281
2    415.761281
3    415.761281
4    415.761281
5    415.761281
dtype: float64

In [70]:
RBC_baseline = RBC_zero * (1 + ((6 / 10) * baseline_folate_pct_increase_from_zero))
RBC_baseline

wealth_quintile
1    415.761281
2    415.761281
3    415.761281
4    415.761281
5    415.761281
dtype: float64

In [71]:
RBC_intervention = RBC_zero * (
    1 + ((6 / 10) * intevention_folate_pct_increase_from_zero)
)
RBC_intervention

scenario      wealth_quintile
intervention  1                  426.349935
              2                  436.409438
              3                  445.060247
              4                  455.784531
              5                  460.758708
dtype: float64

In [72]:
# TODO: Document this function better. I think it's the inverse of the
# backcalc_rbc function above
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p

In [73]:
# NOTE: All rates here are per birth!
s_ntd_affected_pregnancy_rate_zero = calc_ntd_pr(RBC_zero, "crider")
10_000 * s_ntd_affected_pregnancy_rate_zero

wealth_quintile
1    34.095017
2    34.095017
3    34.095017
4    34.095017
5    34.095017
dtype: float64

In [74]:
s_ntd_affected_pregnancy_rate_baseline = calc_ntd_pr(RBC_baseline, "crider")
10_000 * s_ntd_affected_pregnancy_rate_baseline

wealth_quintile
1    34.095017
2    34.095017
3    34.095017
4    34.095017
5    34.095017
dtype: float64

In [75]:
s_ntd_affected_pregnancy_rate_intervention = calc_ntd_pr(RBC_intervention, "crider")
10_000 * s_ntd_affected_pregnancy_rate_intervention

scenario      wealth_quintile
intervention  1                  32.668052
              2                  31.398272
              3                  30.367834
              4                  29.163154
              5                  28.629960
dtype: float64

In [76]:
s_ntd_affected_pregnancies_zero = (
    s_ntd_affected_pregnancy_rate_zero * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_zero

wealth_quintile
1    7596.117757
2    7767.503194
3    7031.727783
4    6232.914051
5    5474.484244
dtype: float64

In [77]:
s_ntd_affected_pregnancies_baseline = (
    s_ntd_affected_pregnancy_rate_baseline * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_baseline

wealth_quintile
1    7596.117757
2    7767.503194
3    7031.727783
4    6232.914051
5    5474.484244
dtype: float64

In [78]:
s_ntd_affected_pregnancies_intervention = (
    s_ntd_affected_pregnancy_rate_intervention * s_births_and_stillbirths_by_wealth
)
s_ntd_affected_pregnancies_intervention

scenario      wealth_quintile
intervention  1                  7278.200535
              2                  7153.132715
              3                  6263.036641
              4                  5331.319634
              5                  4596.984541
dtype: float64

In [79]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_affected_pregnancies_zero.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="zero")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_affected_pregnancies_baseline.rename("value")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        *[
            s_ntd_affected_pregnancies_intervention.loc[intervention_scenario]
            .rename("value")
            .to_frame()
            .assign(entity="ntd", scenario=intervention_scenario)
            .set_index(["entity", "scenario"], append=True)
            .value
            for intervention_scenario in intervention_scenarios
        ],
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            7596.117757
2                ntd     zero            7767.503194
3                ntd     zero            7031.727783
4                ntd     zero            6232.914051
                                            ...     
2                ntd     intervention    7153.132715
3                ntd     intervention    6263.036641
4                ntd     intervention    5331.319634
5                ntd     intervention    4596.984541
Name: value, Length: 15, dtype: float64

In [80]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       317.917222
2                ntd       614.370479
3                ntd       768.691142
4                ntd       901.594417
5                ntd       877.499703
Name: value, dtype: float64

In [81]:
(
    ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "zero"
    ].droplevel("scenario")
    - ntd_cases_by_scenario[
        ntd_cases_by_scenario.index.get_level_values("scenario") == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       0.0
2                ntd       0.0
3                ntd       0.0
4                ntd       0.0
5                ntd       0.0
Name: value, dtype: float64

In [82]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [83]:
ntd_deaths_and_stillbirths_by_scenario = ntd_cases_by_scenario * (
    s_ntd_death_or_stillbirth_count / s_ntd_affected_pregnancies
)

In [84]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "intervention"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       192.680762
2                ntd       372.352814
3                ntd       465.882264
4                ntd       546.431233
5                ntd       531.828099
dtype: float64

In [85]:
(
    ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "zero"
    ].droplevel("scenario")
    - ntd_deaths_and_stillbirths_by_scenario[
        ntd_deaths_and_stillbirths_by_scenario.index.get_level_values("scenario")
        == "baseline"
    ].droplevel("scenario")
)

wealth_quintile  entity
1                ntd       0.0
2                ntd       0.0
3                ntd       0.0
4                ntd       0.0
5                ntd       0.0
dtype: float64

In [86]:
# For calculating YLLs
with gbd_data.quiet_gbd_logs():
    tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [87]:
# NOTE: Treating stillbirths as a death!
yll_per_stillbirth_or_death = float(tmrle.iloc[0])
yll_per_stillbirth_or_death

89.95803974533831

In [88]:
ylls_by_scenario = (
    ntd_deaths_and_stillbirths_by_scenario * yll_per_stillbirth_or_death
).rename("value")
ylls_by_scenario

wealth_quintile  entity  scenario    
1                ntd     zero            414148.385733
2                ntd     zero            423492.501283
3                ntd     zero            383377.246596
4                ntd     zero            339825.075845
                                             ...      
2                ntd     intervention    389996.372062
3                ntd     intervention    341467.391332
4                ntd     intervention    290669.193264
5                ntd     intervention    250632.466206
Name: value, Length: 15, dtype: float64

In [89]:
path = f"./results/{location}/{vehicle}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)